# Tema alesa

### 18. Sisteme automate bazate pe procesarea semnalelor EEG și învățare automată

Actualizat pentru 2026: extins cu BCI bazată pe imaginație motorie pentru controlul unei proteze.
Obiectiv: Obiectivul acestui proiect este dezvoltarea unei interfețe creier-calculator (BCI) care
achiziționează semnale EEG, clasifică intenția utilizatorului și transmite comanda identificată
către un dispozitiv controlat. Aplicația principală propusă este controlul unei proteze de
membru superior prin imaginația motorie a mâinii stângi și a mâinii drepte, cu un model de
clasificare care generalizează la subiecți noi fără calibrare individuală. Sunt acceptate și alte
aplicații (control robotic, conducere autonomă, automatizarea locuinței).

__Setul de date 1 – setul de date de imaginație motorie WBCIC-SHU (Yang et al., 2025)__

Înregistrări EEG multi-subiect și multi-sesiune ale imaginației motorii a mâinii stângi și a
mâinii drepte, cu 53 de canale; utilizat în ediția anterioară pentru definirea protocolului
de antrenare/validare (scenarii intra-sesiune, inter-sesiune și inter-subiect).

__Placa OpenBCI Cyton (Daisy) cu 16 canale, împreună cu casca Ultracortex Mark IV__

achiziție cu BrainFlow (interfețe Python/MATLAB) sau cu OpenBCI GUI.
• Documentație: docs.openbci.com/Cyton/CytonLanding și brainflow.readthedocs.io;
dosarele proiectelor anterioare conțin exemple de înregistrări (Fatigue state, articol
despre imaginație motorie, EEGLAB).

__Metodologie:__ 

Preprocesare în MATLAB/EEGLAB sau Python (filtrare trece-bandă și notch,
re-referențiere, segmentare în epoci, eliminarea liniei de bază, re-eșantionare); analiză
temporală, spațială și spectrală pentru confirmarea calității datelor; compararea CSP + SVM,
FBCSP + SVM, EEGNet și DeepConvNet în scenarii cu dificultate crescândă (intra-sesiune,
inter-sesiune, inter-subiect); selectarea arhitecturii cu cel mai bun compromis între
performanță și cost computațional; validare pe subiecți complet izolați de antrenare; reducerea
numărului de canale pentru a corespunde căștii disponibile; interfață cu utilizatorul
(monitorizarea predicției și a acurateței) și transmiterea comenzii către dispozitiv (de exemplu,
o proteză acționată de servomotoare prin Arduino).

__Rezultate așteptate:__

• Un clasificator de imaginație motorie independent de subiect, validat pe subiecți izolați.
• Un studiu al influenței numărului de canale asupra acurateței.
• Un lanț demonstrativ complet: EEG → preprocesare → clasificator → interfață cu
utilizatorul → proteză/robot.

In [1]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Import necessary libraries

import numpy as np
import pandas as pd
import warnings as wn
import matplotlib.pyplot as plt

from sklearn import *
from pathlib import Path
from moabb.datasets import Yang2025
from moabb.paradigms import LeftRightImagery

In [2]:
# Ignore all warnings
wn.filterwarnings('ignore')

In [4]:
# Create general variables

_data_path = '../data'
_limit_subjects = 3
_data_dir = Path(_data_path)
_train_path = f'{_data_path}/participants.tsv'

In [5]:
# Read the dataset
dataset = Yang2025()
print(f'Available: {dataset.subject_list}')

Available: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51]


In [6]:
# Load some pacients
subjects = dataset.subject_list[:_limit_subjects] 

In [7]:
# Set the paradigm dataset
paradigm = LeftRightImagery(fmin=8, fmax=32)

In [8]:
# Create the dataframes
X, y, metadata = paradigm.get_data(dataset=dataset, subjects=subjects)

In [9]:
# Extract dimensions of input data
n_trials, n_channels, n_timestamps = X.shape

In [10]:
# Swap the second and third dimensions of X
X_swap = np.transpose(X, (0, 2, 1))  

In [11]:
# Reshape the collapse trials and timestamps into rows
X_tr = X_swap.reshape(n_trials * n_timestamps, n_channels)

In [12]:
# Convert the np array to a pandas DataFrame

columns = [f'ch_{i}' for i in range(n_channels)]
X_data = pd.DataFrame(X_tr, columns=columns)
y_data = pd.DataFrame(np.repeat(y, n_timestamps), columns=['label'])

In [13]:
# Add index columns 

X_data.insert(0, 'trial', np.repeat(np.arange(n_trials), n_timestamps))
X_data.insert(1, 'timestamp', np.tile(np.arange(n_timestamps), n_trials))

In [14]:
# Display the first n rows
X_data.head(n=10)

,trial,timestamp,ch_0,ch_1,ch_2,ch_3,ch_4,ch_5,ch_6,ch_7,...,ch_49,ch_50,ch_51,ch_52,ch_53,ch_54,ch_55,ch_56,ch_57,ch_58
0,0,0,-0.002103,-0.001852,-0.002874,-0.003164,-0.001652,-0.001154,-0.000303,-0.002466,...,0.002785,0.001116,0.002950,0.001116,0.002951,0.001198,0.002832,0.003152,0.002958,0.002928
1,0,1,-0.002297,-0.002012,-0.002857,-0.003467,-0.001647,-0.001074,-0.000666,-0.002685,...,0.002728,0.000978,0.002896,0.000986,0.002913,0.001058,0.002794,0.003080,0.002912,0.002866
2,0,2,-0.002465,-0.002167,-0.002790,-0.003735,-0.001618,-0.000980,-0.001025,-0.002871,...,0.002611,0.000829,0.002777,0.000845,0.002809,0.000904,0.002692,0.002942,0.002804,0.002741
3,0,3,-0.002604,-0.002315,-0.002677,-0.003964,-0.001563,-0.000877,-0.001377,-0.003021,...,0.002441,0.000672,0.002603,0.000698,0.002647,0.000742,0.002534,0.002746,0.002643,0.002563
4,0,4,-0.002713,-0.002453,-0.002523,-0.004151,-0.001481,-0.000769,-0.001720,-0.003133,...,0.002231,0.000515,0.002383,0.000551,0.002438,0.000578,0.002331,0.002504,0.002439,0.002342
5,0,5,-0.002791,-0.002581,-0.002331,-0.004293,-0.001374,-0.000662,-0.002052,-0.003204,...,0.001989,0.000363,0.002128,0.000409,0.002192,0.000418,0.002094,0.002227,0.002203,0.002088
6,0,6,-0.002836,-0.002694,-0.002108,-0.004388,-0.001242,-0.000559,-0.002370,-0.003236,...,0.001728,0.000221,0.001851,0.000278,0.001920,0.000266,0.001832,0.001928,0.001945,0.001815
7,0,7,-0.002850,-0.002792,-0.001860,-0.004436,-0.001085,-0.000465,-0.002671,-0.003227,...,0.001458,0.000094,0.001563,0.000162,0.001635,0.000128,0.001557,0.001618,0.001677,0.001532
8,0,8,-0.002832,-0.002872,-0.001593,-0.004435,-0.000906,-0.000383,-0.002954,-0.003180,...,0.001191,-0.000013,0.001275,0.000064,0.001347,0.000008,0.001280,0.001309,0.001411,0.001252
9,0,9,-0.002783,-0.002932,-0.001315,-0.004388,-0.000708,-0.000318,-0.003214,-0.003095,...,0.000936,-0.000097,0.000998,-0.000011,0.001067,-0.000090,0.001012,0.001012,0.001155,0.000984


In [17]:
# Save the dataframes to CSV files
X_data.to_csv(f'{_data_path}/X_data.csv', index=False)
y_data.to_csv(f'{_data_path}/y_data.csv', index=False)

In [18]:
# Show the shape of the data
x_rows, x_cols = X_data.shape
y_rows, y_cols = y_data.shape
print('Input Rows: ', x_rows)
print('Input Columns: ', x_cols)
print('Output Rows: ', y_rows)
print('Output Columns: ', y_cols)

Input Rows:  7201800
Input Columns:  61
Output Rows:  7201800
Output Columns:  1


In [19]:
del X, y, X_swap, X_tr, columns, n_trials, n_channels, n_timestamps

In [20]:
# Describe the dataframe
X_data.describe()

,trial,timestamp,ch_0,ch_1,ch_2,ch_3,ch_4,ch_5,ch_6,ch_7,...,ch_49,ch_50,ch_51,ch_52,ch_53,ch_54,ch_55,ch_56,ch_57,ch_58
count,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,...,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06,7.201800e+06
mean,8.995000e+02,2.000000e+03,-5.037430e-07,-2.774822e-07,-5.665490e-07,-3.505568e-07,-3.176816e-07,-2.765827e-07,-6.410701e-07,4.208841e-08,...,-6.580104e-07,-3.660486e-07,-6.402779e-07,-3.000054e-07,-4.881414e-07,-3.676242e-07,-5.380211e-07,-1.065204e-06,-8.665663e-07,-7.624550e-07
std,5.196152e+02,1.154989e+03,6.080686e-03,6.246648e-03,6.225072e-03,5.200560e-03,5.074073e-03,6.200233e-03,6.188857e-03,4.416911e-03,...,4.222028e-03,5.382193e-03,4.572197e-03,5.384014e-03,4.559760e-03,5.522136e-03,4.660255e-03,5.572468e-03,5.973927e-03,4.838569e-03
min,0.000000e+00,0.000000e+00,-7.442839e-02,-1.626877e-01,-8.390891e-02,-9.734455e-02,-7.653700e-02,-2.002351e-01,-1.006437e-01,-6.846290e-02,...,-7.534014e-02,-9.698726e-02,-7.166976e-02,-9.694946e-02,-7.147738e-02,-8.658303e-02,-7.165454e-02,-8.826613e-02,-9.655799e-02,-7.165472e-02
25%,4.497500e+02,1.000000e+03,-2.806814e-03,-2.938345e-03,-2.907636e-03,-2.488404e-03,-2.380350e-03,-3.097209e-03,-3.054111e-03,-2.122026e-03,...,-1.686351e-03,-2.038509e-03,-1.847790e-03,-2.045959e-03,-1.994107e-03,-2.230675e-03,-2.075118e-03,-2.201954e-03,-2.192923e-03,-2.111911e-03
50%,8.995000e+02,2.000000e+03,-6.235323e-06,1.666160e-06,-2.654487e-06,-1.043365e-05,-1.010386e-05,3.791506e-06,8.404372e-06,-1.833562e-05,...,-2.677393e-06,2.975037e-06,3.675315e-06,2.040896e-06,3.285471e-06,-2.513665e-06,3.014685e-06,-8.554372e-06,-5.205657e-06,-2.703424e-07
75%,1.349250e+03,3.000000e+03,2.810237e-03,2.954790e-03,2.910359e-03,2.478031e-03,2.370552e-03,3.112502e-03,3.068831e-03,2.103069e-03,...,1.692456e-03,2.060564e-03,1.868585e-03,2.067776e-03,2.019630e-03,2.249685e-03,2.099652e-03,2.206857e-03,2.211269e-03,2.131801e-03
max,1.799000e+03,4.000000e+03,9.204475e-02,1.178118e-01,8.226823e-02,8.013634e-02,8.005361e-02,1.488552e-01,1.005164e-01,6.573888e-02,...,5.640355e-02,7.069023e-02,6.071376e-02,7.086175e-02,5.911091e-02,7.183010e-02,5.971731e-02,7.403781e-02,1.160072e-01,6.117025e-02


In [ ]:
# Show more infos about the dataset
X.info()

<class 'pandas.DataFrame'>
RangeIndex: 30591 entries, 0 to 30590
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   onset           30591 non-null  float64
 1   duration        30591 non-null  float64
 2   trial_type      30591 non-null  str    
 3   value           30591 non-null  int64  
 4   sample          30591 non-null  int64  
 5   participant_id  30591 non-null  str    
 6   session_id      30591 non-null  str    
 7   age             30591 non-null  int64  
 8   sex             30591 non-null  str    
 9   hand            30591 non-null  str    
 10  weight          0 non-null      float64
 11  height          0 non-null      float64
 12  group           30591 non-null  str    
 13  species         30591 non-null  str    
 14  bci_experience  30591 non-null  str    
dtypes: float64(4), int64(3), str(8)
memory usage: 3.5 MB


In [ ]:
# Unique class for each column
object_columns = X.select_dtypes(include=['str']).columns
for col in object_columns:
    print(f'{col}: {X[col].nunique()} values \t\t {X[col].unique()}')

trial_type: 2 values 		 <StringArray>
['left_hand', 'right_hand']
Length: 2, dtype: str
participant_id: 51 values 		 <StringArray>
[ 'sub-1', 'sub-10', 'sub-11', 'sub-12', 'sub-13', 'sub-14', 'sub-15',
 'sub-16', 'sub-17', 'sub-18', 'sub-19',  'sub-2', 'sub-20', 'sub-21',
 'sub-22', 'sub-23', 'sub-24', 'sub-25', 'sub-26', 'sub-27', 'sub-28',
 'sub-29',  'sub-3', 'sub-30', 'sub-31', 'sub-32', 'sub-33', 'sub-34',
 'sub-35', 'sub-36', 'sub-37', 'sub-38', 'sub-39',  'sub-4', 'sub-40',
 'sub-41', 'sub-42', 'sub-43', 'sub-44', 'sub-45', 'sub-46', 'sub-47',
 'sub-48', 'sub-49',  'sub-5', 'sub-50', 'sub-51',  'sub-6',  'sub-7',
  'sub-8',  'sub-9']
Length: 51, dtype: str
session_id: 3 values 		 <StringArray>
['ses-0', 'ses-1', 'ses-2']
Length: 3, dtype: str
sex: 2 values 		 <StringArray>
['F', 'M']
Length: 2, dtype: str
hand: 1 values 		 <StringArray>
['R']
Length: 1, dtype: str
group: 1 values 		 <StringArray>
['healthy']
Length: 1, dtype: str
species: 1 values 		 <StringArray>
['human']
Leng